[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_05_classical_ml/ch23_classification_calibration/03_coding_exam_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 23 Coding Exam — Clinical Classification, Calibration & Thresholding (100 Points)

**Book Reference:** Chapter 23, Section 23.7 (Coding Exam)  
**Suggested Time:** 45–55 minutes (Open Book, Closed Solution)

| Task | Classification & Calibration Pipeline Component | Points |
| :--- | :--- | :---: |
| **Task 23.1** | `evaluate_screening_thresholds(y_true, y_prob, thresholds)` (Operating-point sweep DataFrame) | 30 |
| **Task 23.2** | `calibration_audit_bins(y_true, y_prob, n_bins=5)` (Equal-width reliability bins, ECE & Brier) | 35 |
| **Task 23.3** | `train_calibrated_anemia_classifier(mch_df, seed=42)` (Leak-free scaled + Platt-calibrated pipeline) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 23 Exam Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

tutor.activate(chapter="ch23", notebook="coding_exam")


---
## Exam Task 23.1 (30 Points): `evaluate_screening_thresholds(y_true, y_prob, thresholds)`

Build a clinical threshold operating-point sweep table:
1. For each threshold `t` in `thresholds`, classify `y_pred = (y_prob >= t).astype(int)`.
2. Compute `sensitivity`, `specificity`, `precision`, and `f1` (guarding zero denominators cleanly with `0.0`), plus `flagged_pct = 100.0 * mean(y_pred)` (percentage of patients flagged positive on the 0–100 scale).
3. Return a `pd.DataFrame` with exact columns `["threshold", "sensitivity", "specificity", "precision", "f1", "flagged_pct"]` with all values rounded to **4 decimal places**.

In [ ]:
def evaluate_screening_thresholds(y_true, y_prob, thresholds):
    """Evaluate clinical screening metrics across probability thresholds."""
    yt = np.asarray(y_true, dtype=int)
    yprob = np.asarray(y_prob, dtype=float)
    rows = []
    for t in thresholds:
        yp = (yprob >= t).astype(int)
        tp = int(np.sum((yt == 1) & (yp == 1)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1 = 2.0 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0.0
        rows.append({
            "threshold": round(float(t), 4),
            "sensitivity": round(sens, 4),
            "specificity": round(spec, 4),
            "precision": round(prec, 4),
            "f1": round(f1, 4),
            "flagged_pct": round(100.0 * float(np.mean(yp)), 4),
        })
    cols = [
        "threshold", "sensitivity", "specificity",
        "precision", "f1", "flagged_pct",
    ]
    return pd.DataFrame(rows, columns=cols)


---
## Exam Task 23.2 (35 Points): `calibration_audit_bins(y_true, y_prob, n_bins=5)`

Audit probability calibration via equal-width reliability bins:
1. Partition `[0, 1]` into `n_bins` equal-width bins via `np.linspace(0.0, 1.0, n_bins + 1)`. Use half-open `[low, high)` for the first `n_bins - 1` bins and closed `[low, high]` for the last bin so `p == 1.0` is included.
2. Skip empty bins (`count == 0`). For each non-empty bin, compute `mean_pred` (rounded to 4 decimals), `obs_rate` (rounded to 4 decimals), and `count` (`int`).
3. Compute Expected Calibration Error $\text{ECE} = \sum_b (n_b / N) \cdot |\text{obs}_b - \text{pred}_b|$ and `brier_score` (`brier_score_loss`), both rounded to **4 decimals**.
4. Return `{"ece": round4, "brier_score": round4, "bins": DataFrame}` where `bins` has columns `["mean_pred", "obs_rate", "count"]`.

In [ ]:
def calibration_audit_bins(y_true, y_prob, n_bins=5):
    """Audit probability calibration via equal-width bins, ECE & Brier."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_prob, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    ece = 0.0
    n_total = len(yt)
    for idx in range(n_bins):
        lo, hi = edges[idx], edges[idx + 1]
        if idx < n_bins - 1:
            mask = (yp >= lo) & (yp < hi)
        else:
            mask = (yp >= lo) & (yp <= hi)
        cnt = int(np.sum(mask))
        if cnt == 0:
            continue
        mp = float(np.mean(yp[mask]))
        obs = float(np.mean(yt[mask]))
        ece += (cnt / n_total) * abs(obs - mp)
        rows.append({
            "mean_pred": round(mp, 4),
            "obs_rate": round(obs, 4),
            "count": cnt,
        })
    bins_df = pd.DataFrame(rows, columns=["mean_pred", "obs_rate", "count"])
    return {
        "ece": round(ece, 4),
        "brier_score": round(float(brier_score_loss(yt, yp)), 4),
        "bins": bins_df,
    }


---
## Exam Task 23.3 (35 Points): `train_calibrated_anemia_classifier(mch_df, seed=42)`

Build a leak-free, Platt-calibrated maternal anemia classifier (without mutating `mch_df`):
1. Copy `mch_df`, clean `residence` casing (`str.strip().str.title()`), replace sentinel `anc_visits == 999` with `np.nan`, encode `is_urban = (residence == "Urban").astype(int)`, and map `wealth_quintile` (`Poorest`->1, `Poorer`->2, `Middle`->3, `Richer`->4, `Richest`->5) into `wealth_num`.
2. Drop rows with missing values across `["maternal_anemia", "maternal_age_years", "anc_visits", "wealth_num", "is_urban"]` (do NOT include `maternal_hb_g_dl` — that would leak the diagnostic target!).
3. Run a stratified 75/25 split (`test_size=0.25, stratify=y, random_state=seed`).
4. Fit `pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=500))])` on `(X_train, y_train)` for `prob_raw`, and fit `CalibratedClassifierCV(pipe, method="sigmoid", cv=5)` on `(X_train, y_train)` for `prob_cal`.
5. Return `{"roc_auc": round4, "pr_auc": round4, "brier_raw": round4, "brier_calibrated": round4, "youden_threshold": round4}` (where `roc_auc`, `pr_auc`, `brier_calibrated`, and `youden_threshold` come from `prob_cal`, and `brier_raw` comes from `prob_raw`).

In [ ]:
def train_calibrated_anemia_classifier(mch_df, seed=42):
    """Train leak-free scaled + sigmoid-calibrated anemia classifier."""
    work = mch_df.copy()
    work["residence"] = (
        work["residence"].astype(str).str.strip().str.title()
    )
    work["anc_visits"] = work["anc_visits"].replace(999, np.nan)
    work["is_urban"] = (work["residence"] == "Urban").astype(int)
    wealth_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    work["wealth_num"] = work["wealth_quintile"].map(wealth_map)
    feat_cols = [
        "maternal_age_years", "anc_visits", "wealth_num", "is_urban"
    ]
    clean = work[["maternal_anemia"] + feat_cols].dropna()
    X = clean[feat_cols]
    y = clean["maternal_anemia"].astype(int)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=seed
    )
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500)),
    ])
    pipe.fit(X_tr, y_tr)
    prob_raw = pipe.predict_proba(X_te)[:, 1]
    cal = CalibratedClassifierCV(pipe, method="sigmoid", cv=5)
    cal.fit(X_tr, y_tr)
    prob_cal = cal.predict_proba(X_te)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_te, prob_cal)
    best_idx = int(np.argmax(tpr - fpr))
    return {
        "roc_auc": round(float(roc_auc_score(y_te, prob_cal)), 4),
        "pr_auc": round(float(average_precision_score(y_te, prob_cal)), 4),
        "brier_raw": round(float(brier_score_loss(y_te, prob_raw)), 4),
        "brier_calibrated": round(
            float(brier_score_loss(y_te, prob_cal)), 4
        ),
        "youden_threshold": round(float(thresholds[best_idx]), 4),
    }


In [ ]:
exam_submissions = {
    "ch23_exam_01": evaluate_screening_thresholds,
    "ch23_exam_02": calibration_audit_bins,
    "ch23_exam_03": train_calibrated_anemia_classifier,
}

tutor.check_exam("ch23", exam_submissions)


---
## Chapter 23 Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 23.1 (`flagged_pct` Scale & Extreme Thresholds):** Multiplying `mean(y_pred)` by `100.0` converts the flagged fraction into a clinician-ready percentage (`0–100`). At high thresholds like `0.95` where zero patients are flagged, guarding `tp + fp == 0` avoids `ZeroDivisionError`.
- **Task 23.2 (Right-Endpoint Boundary `p == 1.0`):** If every bin uses half-open `[lo, hi)`, then an exact probability `1.0` falls outside all bins and silently lowers `sum(count)` below `N`! Making only the final bin closed `[lo, 1.0]` guarantees every observation is counted exactly once.
- **Task 23.3 (Preventing Target Leakage & Fitting Calibration Cleanly):** Including `maternal_hb_g_dl` to predict `maternal_anemia` produces a bogus ~0.99 ROC-AUC because WHO defines anemia as `Hb < 11.0 g/dL`. Fitting `CalibratedClassifierCV(pipe, method="sigmoid", cv=5)` strictly on `(X_tr, y_tr)` keeps calibration completely leak-free.